In [3]:
import requests
import base64

url1= "https://api.github.com/repos/zju3dv/pvnet/contents/run.py?ref=master"
response = requests.get(url1)

data = response.json()
encoded_content = data['content']
decoded_content = base64.b64decode(encoded_content).decode("utf-8")
print(decoded_content)

from lib.utils.config import cfg
from lib.utils.arg_utils import args
import torch


def run_gen_mask():
    # OcclusionLindModDB: we need masks for each object
    from lib.utils.data_utils import OcclusionLineModDB, OcclusionLineModDBSyn
    db = OcclusionLineModDB()
    db.get_masks()


if __name__ == '__main__':
    globals()['run_' + args.type]()




In [4]:
import requests
import base64

url1= "https://api.github.com/repos/zju3dv/pvnet/contents/"
response = requests.get(url1)

data=response.json()
print(data)
for content in data:
    if content=='content':
        print(True)
    print(content)


[{'name': '.gitignore', 'path': '.gitignore', 'sha': '997dd936f1ac1705011ccc3e82a07a7b32ce0937', 'size': 418, 'url': 'https://api.github.com/repos/zju3dv/pvnet/contents/.gitignore?ref=master', 'html_url': 'https://github.com/zju3dv/pvnet/blob/master/.gitignore', 'git_url': 'https://api.github.com/repos/zju3dv/pvnet/git/blobs/997dd936f1ac1705011ccc3e82a07a7b32ce0937', 'download_url': 'https://raw.githubusercontent.com/zju3dv/pvnet/master/.gitignore', 'type': 'file', '_links': {'self': 'https://api.github.com/repos/zju3dv/pvnet/contents/.gitignore?ref=master', 'git': 'https://api.github.com/repos/zju3dv/pvnet/git/blobs/997dd936f1ac1705011ccc3e82a07a7b32ce0937', 'html': 'https://github.com/zju3dv/pvnet/blob/master/.gitignore'}}, {'name': 'LICENSE', 'path': 'LICENSE', 'sha': 'd645695673349e3947e8e5ae42332d0ac3164cd7', 'size': 11358, 'url': 'https://api.github.com/repos/zju3dv/pvnet/contents/LICENSE?ref=master', 'html_url': 'https://github.com/zju3dv/pvnet/blob/master/LICENSE', 'git_url': '

In [5]:
import requests
import base64

url1= "https://api.github.com/repos/zju3dv/pvnet/contents/assets/cat.png?ref=master"
response = requests.get(url1)

data=response.json()
for content in data:
    if content=='content':
        print(data['content'])
    print(content)

name
path
sha
size
url
html_url
git_url
download_url
type
...

content
encoding
_links


In [6]:
from pydantic import BaseModel, Field
from typing import Optional
import base64
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, function_tool
import os

github_access_token=os.getenv('Github_access_token')
HEADERS = {"Authorization": github_access_token}

class Navigate_repo_class(BaseModel):
    list_url: Optional[list[dict]]
    content : Optional[str]

@function_tool
def Navigate_repo(url:str, file_type : str)-> Navigate_repo_class:
    '''
    When to call this function->
    One you have to navigate a tree or to get the content of blob

    What to pass in this function->
    1. url : GitHub API contents URL only. Must follow this format:
         https://api.github.com/repos/{user}/{repo}/contents/{path}
         
         NOT the HTML url (https://github.com/...) 
         Use URLs returned by return_file_structure or previous Navigate_repo calls.
         Never construct URLs manually.

    2. file_type : the type of the file the url is pointing towards (blob/tree)
    
    What does the function return->
    list_url : list of the child URLs present within the given url
    content : if the url file_type is a blob then return the content

    What to do next after calling this function->
    file_type='tree' → make parallel Navigate_repo calls on each URL in list_url
    file_type='blob' → pass content to the Parent agent, continue remaining trees
    '''


    response=requests.get(url, headers=HEADERS)
    Data=response.json()
    
    content=None
    response_list=[]

    if file_type=='tree':
        for data in Data:
            response_list.append({'name' : data.get('name'), 'URL' : data.get('url')})

    if file_type=='blob':
        content=base64.b64decode(Data['content']).decode("utf-8")

    return Navigate_repo_class(list_url=response_list, content=content)

@function_tool
def return_file_structure(user: str, repository_name:str):
    '''
    When to call this function->
    After reading the Readme of the repository to understand the repository structure

    What to pass in this function->
    1. user : owner of the repository
    2. repository_name : repository name as given on GitHub
    
    What does the function return->
    returns the string of the repository structure e.g. (Readme.md Blob \n asset tree)

    What to do next after calling this function->
    Use the tool Navigate_repo to explore and understand the repo
    '''

    repo_url=f'https://api.github.com/repos/{user}/{repository_name}/git/trees/master?recursive=1'
    
    response=requests.get(repo_url, headers=HEADERS)
    data=response.json() 

    structure_string=''

    for item in data["tree"]:
        structure_string+=f'''{item['path']}  {item['type']} \n'''

    return structure_string

@function_tool
def get_readme(user:str, repository_name:str):
    '''
    When to call->
    Always call this FIRST before any other tool.

    What to pass in->
    1. user : GitHub username or org name e.g. (https://github.com/zju3dv/pvnet here name is zju3dv)
    2. repository_name : exact repository name as on GitHub e.g. (https://github.com/zju3dv/pvnet here repository_name is pvnet)

    What does it return->
    str : full decoded README content, or 'Readme not found' if absent.

    What to do next->
    Call return_file_structure to map the full repo, using README 
    as context for which files matter.    
    '''

    url=f'''https://api.github.com/repos/{user}/{repository_name}/contents/'''

    response=requests.get(url, headers=HEADERS)

    Data=response.json()

    content='Readme not found'
    for data in Data:
        readme_check=data['name'].lower()
        if readme_check=='readme.md':
            url=data['url']
            result=requests.get(url, headers=HEADERS)
            result=result.json()
            content=result['content']
            content=base64.b64decode(content).decode("utf-8")

    return content

In [7]:

parent_agent_instruction='''
You are a technical educator who creates in-depth tutorials for GitHub repositories.
You have three tools available to explore a repository. You MUST follow this exact 
workflow — do not skip any step.

═══════════════════════════════════════
STEP 1 — Read the README (MANDATORY FIRST STEP)
═══════════════════════════════════════
Call get_readme() first, always.
- Understand the project's purpose, architecture, and key components.
- Note every major feature, library, and module mentioned.
- This is your reference map for all subsequent decisions.

═══════════════════════════════════════
STEP 2 — Get the full file structure (MANDATORY SECOND STEP)
═══════════════════════════════════════
Call return_file_structure() immediately after reading the README.
- Study every path and identify which trees and blobs are relevant.
- Mark ALL files that likely contain core logic based on the README.
- Do NOT proceed to Step 3 until you have a clear picture of what to explore.

═══════════════════════════════════════
STEP 3 — Navigate and read ALL relevant files (MANDATORY — DO NOT SKIP)
═══════════════════════════════════════
You MUST call Navigate_repo() to read file contents. This is not optional.
Skipping this step means you have no actual code to reference — your tutorial 
will be incomplete and unacceptable.

Follow this process:
a) For every relevant directory (type=tree) → call Navigate_repo(url, 'tree')
   to get its children. You can call multiple trees in parallel.

b) For every relevant file (type=blob) → call Navigate_repo(url, 'blob')
   to get its content. You can call multiple blobs in parallel.

c) Prioritize files in this order:
   1. Entry point files (main.py, app.py, index.py, run.py etc.)
   2. Core logic files referenced in the README
   3. Configuration files (requirements.txt, config.py, .env.example)
   4. Utility/helper files that support the core logic
   5. Skip: LICENSE, .gitignore, __pycache__, test files, migration files

d) After reading each file, check for imports or references to other files
   you haven't read yet — navigate those too.

e) Read at minimum every file that contributes to the core functionality.
   Do not stop after 1-2 files.

YOU MUST NOT MOVE TO STEP 4 UNTIL YOU HAVE CALLED Navigate_repo() 
AT LEAST ONCE AND READ THE CORE FILES.

═══════════════════════════════════════
STEP 4 — Write the tutorial (ONLY after Step 3 is complete)
═══════════════════════════════════════
Write a tutorial of AT LEAST 2000 words following this EXACT section order:

─────────────────────────────────────
SECTION 1 — OVERVIEW
─────────────────────────────────────
- What problem does this project solve?
- Who is it for?
- What is the high-level approach — how does it solve the problem?
- What libraries are used and WHY was each chosen over alternatives?
- Keep this engaging — a reader should immediately understand the value
  of this project after reading this section.

─────────────────────────────────────
SECTION 2 — REPOSITORY STRUCTURE
─────────────────────────────────────
- Show the directory tree (use the output of return_file_structure)
- Annotate every file and folder with a one-line description of its role:

  ```
  repo/
  ├── main.py          # entry point — parses args and launches the pipeline
  ├── model/
  │   ├── network.py   # defines the neural network architecture
  │   └── loss.py      # custom loss functions
  ├── utils/
  │   └── helpers.py   # shared utility functions
  └── requirements.txt # project dependencies
  ```
- Do NOT just list file names — every line must have a purpose annotation.

─────────────────────────────────────
SECTION 3 — INSTALLATION
─────────────────────────────────────
- Prerequisites (Python version, CUDA, OS requirements if any)
- Step-by-step install commands from requirements.txt or setup.py
- Environment variables required with explanation of what each does
- Common install pitfalls and how to avoid them

─────────────────────────────────────
SECTION 4 — RUNNING THE PROJECT
─────────────────────────────────────
- The exact command(s) to run the project
- What each CLI argument or config option does
- What the expected output looks like
- Any modes (train vs test, dev vs prod) and how to switch between them

─────────────────────────────────────
SECTION 5 — CODE ARCHITECTURE
─────────────────────────────────────
- How the codebase is organized at a high level
- The design patterns used (e.g. pipeline, agent loop, MVC)
- How data flows through the system from input to output:

  User Input → Module A → Module B → Module C → Final Output

- Why the code is structured this way — what problem does this 
  architecture solve?

─────────────────────────────────────
SECTION 6 — KEY FILES EXPLANATION
─────────────────────────────────────
This is the most detailed section. For every core file you read:

  ### filename.py
  **Purpose:** one sentence on what this file does

  Walk through each key function in the file:

  ### function_name()
  ```python
  # actual code snippet from the file — do not invent or paraphrase code
  ```
  - What this function does
  - Why it is implemented this way
  - How it connects to other parts of the project
  - Any non-obvious logic that needs explanation

Repeat for every core file. Do not skip files you read in Step 3.

─────────────────────────────────────
SECTION 7 — EXAMPLE WORKFLOW
─────────────────────────────────────
- Walk through one complete, concrete end-to-end example
- Use a real or realistic input (not "foo/bar" placeholders)
- Trace exactly what happens at each stage of the code:
  "When you run X, function Y is called, which does Z, then passes 
   the result to function W, which..."
- Show the intermediate states and the final output
- This section should make the reader feel like they watched the 
  code run in front of them

─────────────────────────────────────
SECTION 8 — CUSTOMIZATION
─────────────────────────────────────
- What can be changed without breaking the project
- Config values, hyperparameters, or flags the user can tweak
- How to swap components (e.g. replace the model, change the dataset)
- How to extend the project with new features
- Any hooks or extension points the author built in

─────────────────────────────────────
SECTION 9 — TROUBLESHOOTING
─────────────────────────────────────
- The most common errors a user will hit and exactly how to fix them
- Focus on errors tied to actual code you read — not generic advice
- Format each as:

  **Problem:** description of the error or symptom
  **Cause:** why it happens (reference the relevant code)
  **Fix:** exact steps to resolve it

QUALITY REQUIREMENTS:
- Follow the section order above exactly — do not reorder or merge sections
- Every claim about the code MUST reference actual code read via Navigate_repo
- Minimum 2000 words
- At least 5 actual code snippets from the repository — never invent code
- No guessing — if you did not read a file, do not reference it
- Write for an intermediate developer — technical but clear
'''

In [8]:
import os
import openai
from openai import AsyncOpenAI
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent

In [9]:
from pathlib import Path
from rich.console import Console
console=Console()
load_dotenv()
client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

model = OpenAIChatCompletionsModel(
    model="gpt-4o-mini",
    openai_client=client)

tools=[get_readme, return_file_structure, Navigate_repo]
Parent_Agent=Agent(name='Parent Agent', instructions=parent_agent_instruction, tools=tools, model=model)

message='explain this repo https://github.com/soumil2334/Youtube-Ask-Feature'
repo_text=[]
with trace('GitHub Repo Explainer'):
    result = Runner.run_streamed(starting_agent=Parent_Agent, input=message)
    async for event in result.stream_events():
        if event.type=='raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end='')
            repo_text.append(event.data.delta)

REPO=Path('repo_text')
REPO.mkdir(parents=True, exist_ok=True)
text_file=REPO/'text1.txt'
repo=''.join(repo_text)

from save_as_pdf import save_as_pdf

save_as_pdf(repo, 'Repo_pdf.pdf')


# YouTube Ask Feature Repository Overview

## SECTION 1 — OVERVIEW

**What problem does this project solve?**
The YouTube Ask Feature project, named **AskTube**, allows users to interactively chat with any public YouTube video. By leveraging AI, it helps users seamlessly engage by extracting context from the video and enabling a real-time chat interface.

**Who is it for?**
This project is primarily targeted at developers interested in creating interactive AI-driven applications, educators who want to enhance learning through video content, and general users looking for a comprehensive way to engage with video material.

**What is the high-level approach — how does it solve the problem?**
AskTube operates by downloading a specified YouTube video, transcribing its audio using OpenAI's Whisper model, and subsequently indexing this transcription into ChromaDB. The transcription data can then be queried in real-time through a chat interface, thus enabling users to receive context-driven re

NameError: name 'Markdown' is not defined

In [12]:
with open('repo.txt', 'w', encoding='utf-8') as f:
    f.write(repo)


In [ ]:
from rich.markdown import Markdown
console.print(Markdown(repo))

Here's a detailed overview and walkthrough of the Youtube-Ask-Feature repository, which implements a fascinating   
ability to engage with YouTube videos using AI. This repository allows users to chat with the content of any public
YouTube video by leveraging transcription services and AI models.                                                  

SECTION 1 — OVERVIEW                                                                                               

What problem does this project solve?                                                                              

The Youtube-Ask-Feature repo addresses the challenge of understanding and extracting information from lengthy      
YouTube videos. Users often struggle to find specific information within long videos, leading to frustration and   
wasted time. By transcribing video content and allowing users to interact with it conversationally, this tool      
facilitates easier access to information, making video content more manageable and digestible.                     

Who is it for?                                                                                                     

This project is particularly useful for educators, students, researchers, and anyone who consumes a large amount of
video content and seeks a more dynamic method to engage with it. It's also beneficial for developers interested in 
building conversational AI systems and those looking to implement multimedia content processing applications.      

What is the high-level approach — how does it solve the problem?                                                   

The approach is to allow users to input a public YouTube URL, from which the application will:                     

 1 Download the video using yt-dlp or pytube.                                                                      
 2 Transcribe the video audio to text using OpenAI's Whisper.                                                      
 3 Store the indexed transcript in a database (ChromaDB).                                                          
 4 Use a FastAPI backend to expose an interactive WebSocket chat interface, where users can ask questions regarding
   the content of the video.                                                                                       

By directly interacting with the transcription stored in a backend database, users receive real-time responses     
tailored to their queries.                                                                                         

What libraries are used and WHY was each chosen over alternatives?                                                 

 • FastAPI: Chosen for its simplicity and performance advantages in creating APIs. It supports asynchronous        
   processing, allowing for fast responses.                                                                        
 • OpenAI Whisper: Utilized for transcription because of its high accuracy in converting speech to text, even in   
   noisy environments.                                                                                             
 • ChromaDB: Selected for managing and querying the transcriptions because it supports vector storage and retrieval
   which are essential for handling AI context efficiently.                                                        
 • LangGraph: Employed for managing the logic of the chatbot, allowing for more complex workflows and              
   decision-making processes based on user interactions.                                                           
 • Redis: Used for caching chat history and decision states to facilitate a seamless user experience, ensuring that
   context is preserved.                                                                                           

SECTION 2 — REPOSITORY STRUCTURE                                                                                   

Here’s a breakdown of the repository struct

In [ ]:
with open('repo.txt', 'r', encoding='utf-8') as f:
    a=f.read()

In [ ]:
from save_as_pdf import save_as_pdf
save_as_pdf(a, 'pdf1.pdf')


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'C:\Program Files\Tesseract-OCR\libgobject-2.0-0.dll': error 0x7e